# Example: WT attribution, SEAM foreground, and per-method annotations

One sequence, top to bottom: the raw **WT attribution** map, the **foreground_scaled** map we annotate, then one **annotation track per method** (merged called regions).

Annotations are model-free, so this runs on CPU without waiting for the GPU benchmark job.

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt

BENCH = '/grid/koo/home/pmantill/projects/AutoSituate/k562_agft_benchmark'
EI = '/grid/koo/home/pmantill/projects/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra/eigen-interactions'
for p in (BENCH, EI):
    if p not in sys.path:
        sys.path.insert(0, p)

from fast_logo import fast_logo
import config as C, data, annotate as A, metrics as M
from run_benchmark import annotate_all, METHODS
print('methods:', METHODS)

methods: ['recursive_seqlets', 'tfmodisco_seqlets', 'modiscolite_seqlets', 'modisco_fimo', 'naive', 'jaspar_fimo']


In [2]:
# Load all 84 foregrounds and annotate them (pooled seqlet callers + discovery).
# Runs once; M4 TF-MoDISco discovery is the slow step (~1-2 min).
recs = data.load_dataset()
bins, _ = data.activity_bins(recs)
tracks = A.projected_tracks(recs)
regions = annotate_all(recs, tracks)
id2i = {r['id']: i for i, r in enumerate(recs)}
print(f'{len(recs)} sequences annotated')

  [recursive_seqlets] 59 regions total (6.2s)
  [tfmodisco_seqlets] 96 regions total (0.2s)
  [modiscolite_seqlets] 101 regions total (2.5s)
  [modisco_fimo] running TF-MoDISco discovery ...
  [modisco_fimo] discovered 0 patterns
  [modisco_fimo] 0 regions total (0.0s)
  [naive] 284 regions total (0.0s)
  [jaspar_fimo] 5539 regions total (2.2s)
84 sequences annotated


In [3]:
# Pick the example. Set EXAMPLE_ID to any seq_id to override; default = the
# activating sequence where the most methods fire (richest illustration).
EXAMPLE_ID = None
if EXAMPLE_ID is None:
    cand = [i for i, b in enumerate(bins) if b == 'activating']
    EXAMPLE_ID = recs[max(cand, key=lambda i: sum(bool(regions[m][i]) for m in METHODS))]['id']
print('example:', EXAMPLE_ID, '| bin:', bins[id2i[EXAMPLE_ID]],
      '| true:', round(recs[id2i[EXAMPLE_ID]]['true'], 3))

example: peak13675 | bin: activating | true: 1.259


In [4]:
i = id2i[EXAMPLE_ID]
rec = recs[i]
L = C.L_VAR
wt = np.load(C.FOREGROUNDS / EXAMPLE_ID / 'wt_attribution.npy').astype(float)  # (200,4)
fg = rec['fg'].astype(float)                                                   # (200,4)

# one distinct color per method (no recycling)
COLORS = {
    'recursive_seqlets':   '#1f77b4',
    'tfmodisco_seqlets':   '#ff7f0e',
    'modiscolite_seqlets': '#2ca02c',
    'modisco_fimo':        '#d62728',
    'naive':               '#9467bd',
    'jaspar_fimo':         '#8c564b',
}

fig = plt.figure(figsize=(14, 7))
gs = fig.add_gridspec(3, 1, height_ratios=[1.4, 1.4, 2.2], hspace=0.4)

ax_wt = fig.add_subplot(gs[0])
fast_logo(wt, ax_wt)
ax_wt.set_title(f'{EXAMPLE_ID}   (bin: {bins[i]}, true={rec["true"]:.2f})   —   WT attribution', fontsize=11)

ax_fg = fig.add_subplot(gs[1], sharex=ax_wt)
fast_logo(fg, ax_fg)
ax_fg.set_title('foreground_scaled  (map being annotated)', fontsize=11)

ax = fig.add_subplot(gs[2], sharex=ax_wt)
for yi, m in enumerate(METHODS):
    regs = M.merge_regions(regions[m][i])
    if regs:
        ax.broken_barh([(s, e - s) for s, e in regs], (yi - 0.4, 0.8),
                       facecolors=COLORS[m], edgecolors='none')
    bp = sum(e - s for s, e in regs)
    ax.text(L + 1, yi, f'{len(regs)} regions, {bp}bp', va='center', fontsize=8)
ax.set_yticks(range(len(METHODS)))
ax.set_yticklabels(METHODS)
ax.set_ylim(-0.6, len(METHODS) - 0.4)
ax.invert_yaxis()
ax.set_xlim(0, L)
ax.set_xlabel('position (bp) in the 200bp variable window')
ax.set_title('annotations per method (merged called regions)', fontsize=11)
ax.grid(axis='x', alpha=0.2)

out = C.RESULTS / 'figures' / f'example_{EXAMPLE_ID}.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
print('saved', out)
plt.show()

saved /grid/koo/home/pmantill/projects/AutoSituate/k562_agft_benchmark/results/figures/example_peak13675.png
